In [ ]:
# Install the required GLiNER package
!pip install gliner

In [ ]:
import os
import json
import random
import string
import torch
from gliner import GLiNER
from tqdm.auto import tqdm

# ==========================================
# 1. Google Drive Setup
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# Define your project folder in Google Drive.
PROJECT_DIR = '/content/drive/MyDrive/NER_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================
# 2. Load GLiNER Model
# ==========================================
print("Loading GLiNER Multilingual model... This will take a moment.")

# Use the exact Hugging Face repository string: "urchade" instead of "knowledgator"
model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

if torch.cuda.is_available():
    model.to("cuda")
print("Model loaded successfully!")

Loading GLiNER Multilingual model... This will take a moment.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
# ==========================================
# 3. Helper Functions
# ==========================================
def get_batch_llm_predictions(texts):
    """Uses GLiNER to extract entities for a batch of sentences."""
    allowed_labels = [
        "PERSON", "ORG", "LOCATION", "DATETIME", "MONEY", "EVENT", "NORP"
    ]
    # Use batch_predict_entities instead of predict_entities
    return model.batch_predict_entities(texts, allowed_labels)

def generate_result_id(length=10):
    """Generates a random alphanumeric ID for Label Studio annotations."""
    characters = string.ascii_letters + string.digits
    return ''.join(random.choices(characters, k=length))

In [ ]:
# ==========================================
# 4. Main Processing Logic
# ==========================================
def process_text_file(input_filename, output_filename, save_freq=50):
    """
    Reads a TXT file line by line, processes with GLiNER,
    and saves to a Label Studio compatible JSON.
    """
    input_path = os.path.join(PROJECT_DIR, input_filename)
    output_path = os.path.join(PROJECT_DIR, output_filename)

    print(f"Reading sentences from {input_path}...")

    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            # Read all lines, strip leading/trailing whitespace, and ignore empty lines
            sentences = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"File {input_path} not found. Please check the path.")
        return

    formatted_dataset = []
    start_index = 0

    # --- CHECKPOINT RESUME LOGIC ---
    if os.path.exists(output_path):
        try:
            with open(output_path, 'r', encoding='utf-8') as f:
                formatted_dataset = json.load(f)
            start_index = len(formatted_dataset)
            print(f"Found existing checkpoint! Resuming from record {start_index} out of {len(sentences)}...")
        except json.JSONDecodeError:
            print("Checkpoint file is corrupted or empty. Starting from scratch...")
            formatted_dataset = []
            start_index = 0

    if start_index >= len(sentences):
        print(f"Dataset {input_filename} is already fully processed. Skipping.")
        return

    # --- PROCESS REMAINING RECORDS ---
    for idx in tqdm(range(start_index, len(sentences)), desc=f"Processing {input_filename}"):
        original_text = sentences[idx]

        llm_entities = get_batch_llm_predictions([original_text])[0]
        results_array = []

        for entity in llm_entities:
            start = entity.get("start")
            end = entity.get("end")
            ent_text = entity.get("text")
            ent_label = entity.get("label")

            if start is not None and end is not None:
                result_obj = {
                    "id": generate_result_id(),
                    "type": "labels",
                    "from_name": "label",
                    "to_name": "text",
                    "origin": "prediction",
                    "value": {
                        "start": start,
                        "end": end,
                        "text": ent_text,
                        "labels": [ent_label]
                    }
                }
                results_array.append(result_obj)

        final_record = {
            "data": {
                "text": original_text
            },
            "predictions": [
                {
                    "model_version": "gliner_multi-v2.1",
                    "result": results_array
                }
            ]
        }

        formatted_dataset.append(final_record)

        # --- PERIODIC SAVE LOGIC ---
        if (idx + 1) % save_freq == 0:
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(formatted_dataset, f, indent=2, ensure_ascii=False)

    # Final save when the loop completes
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(formatted_dataset, f, indent=2, ensure_ascii=False)

    print(f"Saved optimized pre-annotations to {output_path}\n")

In [ ]:
# ==========================================
# 5. Execution
# ==========================================
# Define your text file name and the desired JSON output name
INPUT_TXT = "clean_sentences.txt"
OUTPUT_JSON = "dataset_all_sentences_final.json"

process_text_file(INPUT_TXT, OUTPUT_JSON, save_freq=50)
print("Processing completed successfully.")

Reading sentences from /content/drive/MyDrive/NER_Project/clean_sentences.txt...
Found existing checkpoint! Resuming from record 1350 out of 6488...


Processing clean_sentences.txt:   0%|          | 0/5138 [00:00<?, ?it/s]

Saved optimized pre-annotations to /content/drive/MyDrive/NER_Project/dataset_all_sentences_final.json

Processing completed successfully.
